# Phase 8.9: Complete Modality Suite

**WaveToX Universal Adapters + FluxToAny**

This notebook completes the universal I/O layer for FLUX:
- ✅ ImageToWave / WaveToImage_Universal (3 physics engines)
- ✅ AudioToWave / WaveToAudio (stubs for future)
- ✅ FluxToAny unified model class
- ✅ Builds on Flux-X.flx from Phase 8.8

**3 Physics Engines for Image Generation:**
| Engine | Principle | Visual Style |
|--------|-----------|--------------|
| Gravity | Mass attractors | Smooth gradients |
| Interference | Wave superposition | Ripples, patterns |
| Thermodynamic | Energy minimization | Organic textures |

---

## Cell 1: Clone/Pull FLUX Repository

In [ ]:
%%time
import os
from pathlib import Path

REPO_URL = 'https://github.com/Unseengap/FLUX.git'
ROOT = Path('/kaggle/working/FLUX') if Path('/kaggle').exists() else Path('/content/FLUX')

if ROOT.exists():
    print('  Pulling latest...')
    !cd {ROOT} && git pull
else:
    print('  Cloning repo...')
    !git clone {REPO_URL} {ROOT}

os.chdir(ROOT)
print(f'  Working dir: {os.getcwd()}')

# Install dependencies
!pip install -q -e . 2>/dev/null || pip install -q -r requirements.txt
!pip install -q huggingface_hub

print('  ✓ Dependencies installed')

## Cell 2: Setup Paths & Logger

In [ ]:
import sys
from pathlib import Path

ROOT = Path('/kaggle/working/FLUX') if Path('/kaggle').exists() else Path('/content/FLUX')
ROOT_DIR = ROOT
PHASES_DIR = ROOT / 'phases'
CHECKPOINT_DIR = ROOT / 'checkpoints'
CHECKPOINT_DIR.mkdir(exist_ok=True)

# Add all phase paths
for i in range(1, 10):
    p = PHASES_DIR / f'phase{i}'
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

# Add phase8_9 (if exists)
p89 = PHASES_DIR / 'phase8_9'
if p89.exists() and str(p89) not in sys.path:
    sys.path.insert(0, str(p89))
    print(f'  ✓ phase8_9 found')

# Add phase8_8 (has adapters)
p88 = PHASES_DIR / 'phase8_8'
if p88.exists() and str(p88) not in sys.path:
    sys.path.insert(0, str(p88))
    print(f'  ✓ phase8_8 found')

# Add phase8_5 (fallback - has flx_loader)
p85 = PHASES_DIR / 'phase8_5'
if p85.exists() and str(p85) not in sys.path:
    sys.path.insert(0, str(p85))
    print(f'  ✓ phase8_5 found (fallback for flx_loader)')

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from flux_utils import PhaseLogger, get_device

log = PhaseLogger(phase=8)  # Log to phase8.log (8.9 is extension)
log.separator("Phase 8.9: Complete Modality Suite")

print('  ✓ Paths configured')

## Cell 3: Hardware & Secrets

In [ ]:
import torch
import os

log.cell_start("Cell 3 — Hardware & Secrets")

device = get_device()
print(f'  Device: {device}')

if device == 'cuda':
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'  GPU: {gpu_name} ({vram:.1f} GB)')

# Load secrets (Kaggle → Colab → env fallback)
hf_token = None

# Try Kaggle secrets
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    hf_token = secrets.get_secret('HF_TOKEN')
    print('  ✓ Kaggle secrets loaded')
except:
    pass

# Try Colab secrets
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
        print('  ✓ Colab secrets loaded')
    except:
        pass

# Fall back to env vars
if not hf_token:
    hf_token = os.environ.get('HF_TOKEN')
    if hf_token:
        print('  ✓ Environment variables loaded')

# Clean and set token
if hf_token:
    hf_token = hf_token.strip()
    os.environ['HF_TOKEN'] = hf_token
    print('  ✓ HF_TOKEN set')
else:
    print('  ⚠ HF_TOKEN not found (will try anonymous download)')

log.cell_end("Cell 3 — Hardware & Secrets", "PASS")

## Cell 4: Download Model from HuggingFace

In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path

log.cell_start("Cell 4 — Download Model")

# Try Flux-X.flx first (Phase 8.8), fall back to Flux-beta.flx
flx_path = CHECKPOINT_DIR / 'Flux-X.flx'
fallback_path = CHECKPOINT_DIR / 'Flux-beta.flx'

if flx_path.exists():
    size_mb = flx_path.stat().st_size / 1e6
    print(f'  ✓ Flux-X.flx exists ({size_mb:.1f} MB)')
elif fallback_path.exists():
    flx_path = fallback_path
    size_mb = flx_path.stat().st_size / 1e6
    print(f'  ✓ Flux-beta.flx exists ({size_mb:.1f} MB)')
else:
    # Try downloading Flux-X.flx
    print('  Downloading Flux-X.flx from HuggingFace...')
    try:
        downloaded = hf_hub_download(
            repo_id='UnseenGAP/FLUX',
            filename='checkpoints/Flux-X.flx',
            local_dir=str(ROOT_DIR),
            token=hf_token,
        )
        size_mb = Path(downloaded).stat().st_size / 1e6
        print(f'  ✓ Downloaded Flux-X.flx ({size_mb:.1f} MB)')
    except Exception as e:
        print(f'  ⚠ Flux-X.flx download failed: {e}')
        # Fall back to Flux-beta.flx
        print('  Trying Flux-beta.flx fallback...')
        try:
            downloaded = hf_hub_download(
                repo_id='UnseenGAP/FLUX',
                filename='checkpoints/Flux-beta.flx',
                local_dir=str(ROOT_DIR),
                token=hf_token,
            )
            flx_path = CHECKPOINT_DIR / 'Flux-beta.flx'
            size_mb = Path(downloaded).stat().st_size / 1e6
            print(f'  ✓ Downloaded Flux-beta.flx ({size_mb:.1f} MB)')
        except Exception as e2:
            print(f'  ✗ Fallback also failed: {e2}')
            raise RuntimeError("Cannot proceed without checkpoint")

log.cell_end("Cell 4 — Download Model", "PASS")

## Cell 5: Verify .flx Archive

In [ ]:
log.cell_start("Cell 5 — Verify .flx Archive")

# Import flx_loader (from phase8_8 or phase8_5)
try:
    from flx_loader import verify_flx, get_flx_info, load_flux_from_flx
    print('  ✓ flx_loader imported')
except ImportError as e:
    print(f'  ⚠ Import error: {e}')
    raise

# Verify the archive
success, msg = verify_flx(flx_path)
print(f'  Verification: {msg}')

if success:
    info = get_flx_info(flx_path)
    print(f'  Version: {info["version"]}')
    print(f'  Size: {info["file_size_mb"]:.1f} MB')
    print(f'  Components: {info["components"]}')
    
    if 'metadata' in info and info['metadata']:
        meta = info['metadata']
        print(f'  Source phase: {meta.get("source_phase", "?")}')
        print(f'  Adapters: {meta.get("adapters", [])}')
    
    log.success("FLX archive verified")
else:
    log.error(f"FLX verification failed: {msg}")
    raise RuntimeError(msg)

log.cell_end("Cell 5 — Verify .flx Archive", "PASS")

## Cell 6: Load FLUXLarge Model

In [ ]:
log.cell_start("Cell 6 — Load FLUXLarge Model")

from flx_loader import load_flux_from_flx

# Load the full model
model = load_flux_from_flx(
    path=flx_path,
    device=device,
    verbose=True,
    auto_download=False,  # Already downloaded
)

# Quick stats
stats = model.get_stats()
log.metric("Total params", f"{stats.total_params:,}")
log.metric("Episodic entries", stats.episodic_entries)
log.metric("Field energy", f"{stats.field_energy:.2f}")

log.cell_end("Cell 6 — Load FLUXLarge Model", "PASS")

## Cell 7: Initialize All Adapters (Grid, Text, Image, Audio)

In [ ]:
log.cell_start("Cell 7 — Initialize All Adapters")

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from typing import Union, Optional, Tuple
from dataclasses import dataclass
import math

# ─────────────────────────────────────────────
# Grid Adapters (from Phase 8.8)
# ─────────────────────────────────────────────

class GridToWave(nn.Module):
    """Grid → Wave adapter for ARC-style grids."""
    
    def __init__(self, wave_dim: int = 432, num_colors: int = 10, max_size: int = 30,
                 color_dim: int = 32, pattern_dim: int = 64, device: str = 'cpu'):
        super().__init__()
        self.wave_dim = wave_dim
        self.num_colors = num_colors
        self.device = device
        
        self.color_embed = nn.Embedding(num_colors + 1, color_dim)
        self.pos_embed_h = nn.Embedding(max_size, pattern_dim // 2)
        self.pos_embed_w = nn.Embedding(max_size, pattern_dim // 2)
        
        self.conv1 = nn.Conv2d(color_dim, pattern_dim, 3, padding=1)
        self.conv2 = nn.Conv2d(pattern_dim, pattern_dim, 3, padding=1)
        self.conv3 = nn.Conv2d(pattern_dim, pattern_dim, 3, padding=1)
        
        feature_dim = color_dim + pattern_dim + pattern_dim
        self.to_wave = nn.Linear(feature_dim, wave_dim)
        self.global_pool = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(pattern_dim, wave_dim))
        self.to(device)
    
    def encode(self, grid: Union[Tensor, list], mode: str = 'holistic') -> Tensor:
        if isinstance(grid, list):
            grid = torch.tensor(grid, dtype=torch.long, device=self.device)
        else:
            grid = grid.to(self.device)
        H, W = grid.shape
        
        color_feats = self.color_embed(grid)
        h_pos = self.pos_embed_h(torch.arange(H, device=self.device))
        w_pos = self.pos_embed_w(torch.arange(W, device=self.device))
        pos_feats = torch.cat([h_pos.unsqueeze(1).expand(-1, W, -1),
                               w_pos.unsqueeze(0).expand(H, -1, -1)], dim=-1)
        
        x = color_feats.permute(2, 0, 1).unsqueeze(0)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        pattern_feats = x.squeeze(0).permute(1, 2, 0)
        
        if mode == 'holistic':
            return self.global_pool(x).squeeze(0)
        else:
            cell_feats = torch.cat([color_feats, pos_feats, pattern_feats], dim=-1)
            return self.to_wave(cell_feats.view(H * W, -1))
    
    def encode_pair(self, input_grid, output_grid) -> Tuple[Tensor, Tensor, Tensor]:
        in_wave = self.encode(input_grid, mode='holistic')
        out_wave = self.encode(output_grid, mode='holistic')
        return in_wave, out_wave, out_wave - in_wave


class WaveToGrid(nn.Module):
    """Wave → Grid adapter."""
    
    def __init__(self, wave_dim: int = 432, num_colors: int = 10, max_size: int = 30,
                 hidden_dim: int = 256, device: str = 'cpu'):
        super().__init__()
        self.num_colors = num_colors
        self.max_size = max_size
        self.hidden_dim = hidden_dim
        self.device = device
        
        self.wave_proj = nn.Linear(wave_dim, hidden_dim)
        self.size_head = nn.Sequential(nn.Linear(wave_dim, 64), nn.ReLU(), nn.Linear(64, 2))
        self.spatial_expand = nn.Linear(hidden_dim, max_size * max_size * hidden_dim // 4)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(hidden_dim // 4, hidden_dim // 2, 3, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(hidden_dim // 2, num_colors, 3, padding=1))
        self.to(device)
    
    def decode(self, wave: Tensor, grid_size: Optional[Tuple[int, int]] = None) -> Tensor:
        wave = wave.to(self.device)
        if wave.dim() == 1:
            wave = wave.unsqueeze(0)
        
        if grid_size is None:
            size_pred = self.size_head(wave)
            H = int(torch.clamp(size_pred[0, 0], 1, self.max_size).item())
            W = int(torch.clamp(size_pred[0, 1], 1, self.max_size).item())
        else:
            H, W = grid_size
        
        hidden = self.wave_proj(wave)
        spatial = self.spatial_expand(hidden).view(1, self.hidden_dim // 4, self.max_size, self.max_size)
        logits = self.decoder(spatial)[:, :, :H, :W]
        return logits.argmax(dim=1).squeeze(0)


# ─────────────────────────────────────────────
# Text Adapters (wraps CSE)
# ─────────────────────────────────────────────

class TextToWave(nn.Module):
    """Text → Wave adapter (wraps CSE)."""
    
    def __init__(self, wave_dim: int = 432, cse=None, device: str = 'cpu'):
        super().__init__()
        self.wave_dim = wave_dim
        self.cse = cse
        self.device = device
    
    def encode(self, text: str) -> Tensor:
        with torch.no_grad():
            wave = self.cse.encode(text)
        return wave.full.to(self.device)
    
    def encode_pooled(self, text: str) -> Tensor:
        return self.encode(text).mean(dim=0)


class WaveToText(nn.Module):
    """Wave → Text adapter (wraps decoder)."""
    
    def __init__(self, wave_dim: int = 432, decoder=None, device: str = 'cpu'):
        super().__init__()
        self.wave_dim = wave_dim
        self.decoder = decoder
        self.device = device
    
    def decode(self, wave: Tensor, max_length: int = 100) -> str:
        # Use model's generate if available
        if self.decoder is not None and hasattr(self.decoder, 'generate'):
            return self.decoder.generate(wave, max_length=max_length)
        return "[WaveToText requires trained decoder]"


# ─────────────────────────────────────────────
# Style presets for Image Generation
# ─────────────────────────────────────────────

@dataclass
class StylePreset:
    name: str
    gravity: float
    interference: float
    thermodynamic: float

STYLE_PRESETS = {
    'photorealistic': StylePreset('photorealistic', 0.7, 0.2, 0.1),
    'abstract': StylePreset('abstract', 0.2, 0.6, 0.2),
    'crystalline': StylePreset('crystalline', 0.1, 0.3, 0.6),
    'organic': StylePreset('organic', 0.4, 0.1, 0.5),
    'dream': StylePreset('dream', 0.33, 0.33, 0.34),
}


# ─────────────────────────────────────────────
# Image Adapters (3 Physics Engines)
# ─────────────────────────────────────────────

class GravityRenderer(nn.Module):
    """Physics Engine 1: Mass attractors pull colors → smooth gradients."""
    
    def __init__(self, wave_dim=432, num_attractors=16):
        super().__init__()
        self.num_attractors = num_attractors
        self.to_positions = nn.Linear(wave_dim, num_attractors * 2)
        self.to_masses = nn.Linear(wave_dim, num_attractors)
        self.to_colors = nn.Linear(wave_dim, num_attractors * 3)
        self.to_background = nn.Linear(wave_dim, 3)
    
    def forward(self, wave, H, W):
        positions = torch.sigmoid(self.to_positions(wave)).view(self.num_attractors, 2)
        masses = F.softplus(self.to_masses(wave))
        colors = torch.sigmoid(self.to_colors(wave)).view(self.num_attractors, 3)
        background = torch.sigmoid(self.to_background(wave))
        
        y_coords = torch.linspace(0, 1, H, device=wave.device)
        x_coords = torch.linspace(0, 1, W, device=wave.device)
        grid_y, grid_x = torch.meshgrid(y_coords, x_coords, indexing='ij')
        grid = torch.stack([grid_x, grid_y], dim=-1)
        
        image = background.view(1, 1, 3).expand(H, W, 3).clone()
        for i in range(self.num_attractors):
            dist = ((grid - positions[i]) ** 2).sum(dim=-1).sqrt()
            pull = masses[i] / (dist + 0.1) ** 2
            pull = pull / (pull.max() + 1e-6)
            image = image + pull.unsqueeze(-1) * colors[i].view(1, 1, 3)
        
        return (image / (image.max() + 1e-6)).clamp(0, 1)


class InterferenceRenderer(nn.Module):
    """Physics Engine 2: Wave superposition → ripples, patterns."""
    
    def __init__(self, wave_dim=432, num_sources=8):
        super().__init__()
        self.num_sources = num_sources
        self.to_positions = nn.Linear(wave_dim, num_sources * 2)
        self.to_frequencies = nn.Linear(wave_dim, num_sources)
        self.to_phases = nn.Linear(wave_dim, num_sources)
        self.to_amplitudes = nn.Linear(wave_dim, num_sources * 3)
    
    def forward(self, wave, H, W):
        positions = torch.sigmoid(self.to_positions(wave)).view(self.num_sources, 2)
        frequencies = F.softplus(self.to_frequencies(wave)) * 20 + 1
        phases = self.to_phases(wave) * 2 * math.pi
        amplitudes = torch.sigmoid(self.to_amplitudes(wave)).view(self.num_sources, 3)
        
        y_coords = torch.linspace(0, 1, H, device=wave.device)
        x_coords = torch.linspace(0, 1, W, device=wave.device)
        grid_y, grid_x = torch.meshgrid(y_coords, x_coords, indexing='ij')
        grid = torch.stack([grid_x, grid_y], dim=-1)
        
        image = torch.zeros(H, W, 3, device=wave.device)
        for i in range(self.num_sources):
            dist = ((grid - positions[i]) ** 2).sum(dim=-1).sqrt()
            wave_val = (torch.cos(frequencies[i] * dist * 2 * math.pi + phases[i]) + 1) / 2
            image = image + wave_val.unsqueeze(-1) * amplitudes[i].view(1, 1, 3)
        
        return (image / (image.max() + 1e-6)).clamp(0, 1)


class ThermodynamicRenderer(nn.Module):
    """Physics Engine 3: Energy minimization → organic textures."""
    
    def __init__(self, wave_dim=432, settling_steps=10):
        super().__init__()
        self.settling_steps = settling_steps
        self.to_energy = nn.Linear(wave_dim, 64)
        self.to_temperature = nn.Linear(wave_dim, 1)
    
    def forward(self, wave, H, W):
        energy = self.to_energy(wave)
        temp = torch.sigmoid(self.to_temperature(wave))
        
        y_coords = torch.linspace(-1, 1, H, device=wave.device)
        x_coords = torch.linspace(-1, 1, W, device=wave.device)
        grid_y, grid_x = torch.meshgrid(y_coords, x_coords, indexing='ij')
        
        image = torch.zeros(H, W, 3, device=wave.device)
        for i in range(min(16, len(energy) // 4)):
            freq_x, freq_y = (i % 4 + 1), (i // 4 + 1)
            basis = torch.sin(freq_x * grid_x * math.pi) * torch.sin(freq_y * grid_y * math.pi)
            
            r = energy[i * 4].item() if i * 4 < len(energy) else 0
            g = energy[i * 4 + 1].item() if i * 4 + 1 < len(energy) else 0
            b = energy[i * 4 + 2].item() if i * 4 + 2 < len(energy) else 0
            
            image[:, :, 0] += basis * r
            image[:, :, 1] += basis * g
            image[:, :, 2] += basis * b
        
        # Settling iterations
        for step in range(self.settling_steps):
            cooling = 1 - (step / self.settling_steps) * (1 - temp)
            image_padded = F.pad(image.permute(2, 0, 1).unsqueeze(0), (1, 1, 1, 1), mode='reflect')
            kernel = torch.ones(1, 1, 3, 3, device=wave.device) / 9
            blurred = [F.conv2d(image_padded[:, c:c+1], kernel, padding=0) for c in range(3)]
            image_blurred = torch.cat(blurred, dim=1).squeeze(0).permute(1, 2, 0)
            image = cooling * image + (1 - cooling) * image_blurred
        
        return ((image - image.min()) / (image.max() - image.min() + 1e-6)).clamp(0, 1)


class WaveToImage_Universal(nn.Module):
    """Wave → Image with 3 physics-based renderers."""
    
    def __init__(self, wave_dim=432, device='cpu'):
        super().__init__()
        self.wave_dim = wave_dim
        self.gravity = GravityRenderer(wave_dim)
        self.interference = InterferenceRenderer(wave_dim)
        self.thermodynamic = ThermodynamicRenderer(wave_dim)
        self.auto_blend = nn.Sequential(
            nn.Linear(wave_dim, 64), nn.GELU(),
            nn.Linear(64, 3), nn.Softmax(dim=-1)
        )
        self.to(device)
    
    def decode(self, wave, size=(256, 256), style='dream', custom_weights=None, auto_blend=False):
        if wave.dim() == 2:
            wave = wave.mean(dim=0)
        H, W = size
        
        if auto_blend:
            weights = self.auto_blend(wave)
            w_g, w_i, w_t = weights[0], weights[1], weights[2]
        elif custom_weights:
            w_g, w_i, w_t = custom_weights
        else:
            preset = STYLE_PRESETS.get(style, STYLE_PRESETS['dream'])
            w_g, w_i, w_t = preset.gravity, preset.interference, preset.thermodynamic
        
        img_g = self.gravity(wave, H, W)
        img_i = self.interference(wave, H, W)
        img_t = self.thermodynamic(wave, H, W)
        
        return (w_g * img_g + w_i * img_i + w_t * img_t).clamp(0, 1)


# ─────────────────────────────────────────────
# Audio Adapters (Stubs for future)
# ─────────────────────────────────────────────

class AudioToWave(nn.Module):
    """Audio → Wave adapter (STUB)."""
    
    def __init__(self, wave_dim=432, sample_rate=22050, n_mels=80, device='cpu'):
        super().__init__()
        self.wave_dim = wave_dim
        self.sample_rate = sample_rate
        self.n_mels = n_mels
        self.mel_to_wave = nn.Linear(n_mels, wave_dim)
        self.to(device)
    
    def encode(self, audio):
        if audio.dim() == 1:
            num_frames = max(1, audio.shape[0] // 256)
            mel = torch.randn(num_frames, self.n_mels, device=audio.device)
        else:
            mel = audio
        return self.mel_to_wave(mel)


class WaveToAudio(nn.Module):
    """Wave → Audio adapter (STUB)."""
    
    def __init__(self, wave_dim=432, sample_rate=22050, n_mels=80, device='cpu'):
        super().__init__()
        self.wave_dim = wave_dim
        self.sample_rate = sample_rate
        self.hop_length = 256
        self.wave_to_mel = nn.Linear(wave_dim, n_mels)
        self.to(device)
    
    def decode(self, waves, duration_sec=None):
        if waves.dim() == 1:
            waves = waves.unsqueeze(0)
        num_frames = waves.shape[0]
        num_samples = num_frames * self.hop_length
        
        t = torch.linspace(0, num_frames / 100, num_samples, device=waves.device)
        mel = self.wave_to_mel(waves)
        energy = mel.abs().mean(dim=1)
        energy_interp = F.interpolate(energy.view(1, 1, -1), size=num_samples, mode='linear').squeeze()
        
        waveform = energy_interp * torch.sin(2 * 3.14159 * 440 * t)
        return waveform / (waveform.abs().max() + 1e-6)


# ─────────────────────────────────────────────
# Initialize All Adapters
# ─────────────────────────────────────────────

print('\n  Initializing adapters...')

# Grid adapters
grid_to_wave = GridToWave(device=device)
wave_to_grid = WaveToGrid(device=device)
print('  ✓ GridToWave/WaveToGrid initialized')

# Text adapters (wrap CSE)
text_to_wave = TextToWave(device=device, cse=model.cse)
wave_to_text = WaveToText(device=device, decoder=model.decoder if hasattr(model, 'decoder') else None)
print('  ✓ TextToWave/WaveToText initialized (wraps CSE)')

# Image adapters (3 physics engines)
wave_to_image = WaveToImage_Universal(device=device)
print(f'  ✓ WaveToImage_Universal: {sum(p.numel() for p in wave_to_image.parameters()):,} params')

# Audio adapters (stubs)
audio_to_wave = AudioToWave(device=device)
wave_to_audio = WaveToAudio(device=device)
print('  ✓ AudioToWave/WaveToAudio initialized (stubs)')


# ─────────────────────────────────────────────
# FluxToAny Unified Model
# ─────────────────────────────────────────────

@dataclass
class FluxOutput:
    output: object
    wave: torch.Tensor
    input_modality: str
    output_modality: str


class FluxToAny(nn.Module):
    """Universal FLUX interface. Load once, process any modality."""
    
    def __init__(self, flux_model, device='cpu'):
        super().__init__()
        self.flux = flux_model
        self.device = device
        
        # Input adapters
        self.input_adapters = nn.ModuleDict({
            'text': TextToWave(device=device, cse=flux_model.cse if hasattr(flux_model, 'cse') else None),
            'grid': GridToWave(device=device),
            'audio': AudioToWave(device=device),
        })
        
        # Output adapters
        self.output_adapters = nn.ModuleDict({
            'text': WaveToText(device=device, decoder=flux_model.decoder if hasattr(flux_model, 'decoder') else None),
            'grid': WaveToGrid(device=device),
            'image': WaveToImage_Universal(device=device),
            'audio': WaveToAudio(device=device),
        })
    
    def detect_modality(self, x):
        if isinstance(x, str):
            return 'text'
        if isinstance(x, list) and all(isinstance(row, list) for row in x):
            return 'grid'
        if isinstance(x, torch.Tensor):
            if x.dim() == 2 and x.dtype in [torch.long, torch.int]:
                return 'grid'
            if x.dim() == 1:
                return 'audio'
        return 'unknown'
    
    def encode(self, x, modality=None):
        if modality is None:
            modality = self.detect_modality(x)
        adapter = self.input_adapters.get(modality)
        if adapter is None:
            raise ValueError(f"No input adapter for: {modality}")
        return adapter.encode(x) if hasattr(adapter, 'encode') else adapter(x)
    
    def decode(self, wave, modality, **kwargs):
        adapter = self.output_adapters.get(modality)
        if adapter is None:
            raise ValueError(f"No output adapter for: {modality}")
        return adapter.decode(wave, **kwargs) if hasattr(adapter, 'decode') else adapter(wave, **kwargs)
    
    def forward(self, x, output_modality='text', input_modality=None, **kwargs):
        if input_modality is None:
            input_modality = self.detect_modality(x)
        
        wave = self.encode(x, modality=input_modality)
        
        # Pool to single vector
        if wave.dim() == 2:
            wave = wave.mean(dim=0)
        
        output = self.decode(wave, modality=output_modality, **kwargs)
        
        return FluxOutput(
            output=output,
            wave=wave,
            input_modality=input_modality,
            output_modality=output_modality,
        )


# Initialize FluxToAny
flux_to_any = FluxToAny(model, device=device)
print('  ✓ FluxToAny initialized')

print(f'\n  Available adapters:')
print(f'    Input:  text (via CSE), grid, audio (stub)')
print(f'    Output: text, grid, image (3 engines), audio (stub)')
print(f'  Style presets: {list(STYLE_PRESETS.keys())}')

log.cell_end("Cell 7 — Initialize All Adapters", "PASS")

## Cell 8: Test 1 — Grid Round-Trip Encoding

In [ ]:
log.cell_start("Cell 8 — Test 1: Grid Round-Trip")

# Get grid adapters
grid_to_wave = flux_to_any.input_adapters['grid']
wave_to_grid = flux_to_any.output_adapters['grid']

# Test 1: Same grid produces identical wave
grid = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
wave1 = grid_to_wave.encode(grid, mode='holistic')
wave2 = grid_to_wave.encode(grid, mode='holistic')

similarity = F.cosine_similarity(wave1.unsqueeze(0), wave2.unsqueeze(0)).item()
log.info(f"Same grid similarity: {similarity:.4f}")
assert similarity > 0.999, "Same grid should produce identical wave"
log.success("✓ Same grid consistency")

# Test 2: Different grids have different embeddings
grid2 = [[9, 8, 7], [6, 5, 4], [3, 2, 1]]
wave3 = grid_to_wave.encode(grid2, mode='holistic')
diff_sim = F.cosine_similarity(wave1.unsqueeze(0), wave3.unsqueeze(0)).item()
log.info(f"Different grid similarity: {diff_sim:.4f}")
assert diff_sim < 0.99, "Different grids should have different embeddings"
log.success("✓ Different grids discriminable")

# Test 3: Decode produces correct dimensions
for h, w in [(3, 3), (5, 5), (7, 4)]:
    decoded = wave_to_grid.decode(wave1, grid_size=(h, w))
    assert decoded.shape == (h, w), f"Expected {(h, w)}, got {decoded.shape}"
log.success("✓ Decode dimensions correct")

# Test 4: Values in valid range
decoded = wave_to_grid.decode(wave1, grid_size=(5, 5))
assert decoded.min() >= 0 and decoded.max() <= 9, "Values must be 0-9"
log.success("✓ Values in range [0, 9]")

log.cell_end("Cell 8 — Test 1: Grid Round-Trip", "PASS")

## Cell 9: Test 2 — Image Generation Quality

In [ ]:
log.cell_start("Cell 9 — Test 2: Image Generation")

# Create test wave
torch.manual_seed(42)
test_wave = torch.randn(432, device=device)
test_wave = test_wave / test_wave.norm() * 10

# Test 1: All physics engines work
size = (64, 64)

img_g = wave_to_image.gravity(test_wave, size[0], size[1])
assert img_g.shape == (64, 64, 3) and img_g.min() >= 0 and img_g.max() <= 1
log.success("✓ Gravity renderer works")

img_i = wave_to_image.interference(test_wave, size[0], size[1])
assert img_i.shape == (64, 64, 3) and img_i.min() >= 0 and img_i.max() <= 1
log.success("✓ Interference renderer works")

img_t = wave_to_image.thermodynamic(test_wave, size[0], size[1])
assert img_t.shape == (64, 64, 3) and img_t.min() >= 0 and img_t.max() <= 1
log.success("✓ Thermodynamic renderer works")

# Test 2: All style presets
for style_name in STYLE_PRESETS:
    img = wave_to_image.decode(test_wave, size=size, style=style_name)
    assert img.shape == (64, 64, 3), f"Style {style_name} failed"
log.success(f"✓ All {len(STYLE_PRESETS)} style presets work")

# Test 3: Different waves → different images
wave1 = torch.randn(432, device=device)
wave2 = torch.randn(432, device=device)
img1 = wave_to_image.decode(wave1, size=size, style='dream')
img2 = wave_to_image.decode(wave2, size=size, style='dream')
diff = (img1 - img2).abs().mean().item()
assert diff > 0.01, "Different waves should produce different images"
log.success(f"✓ Different waves → different images (diff={diff:.4f})")

# Test 4: Auto-blend produces valid weights
auto_weights = wave_to_image.auto_blend(test_wave)
assert abs(auto_weights.sum().item() - 1.0) < 0.01, "Weights must sum to 1"
assert (auto_weights >= 0).all(), "All weights must be positive"
log.success("✓ Auto-blend valid weights")

log.cell_end("Cell 9 — Test 2: Image Generation", "PASS")

## Cell 10: Test 3 — FluxToAny All Modalities

In [ ]:
log.cell_start("Cell 10 — Test 3: FluxToAny")

# Test modality detection
assert flux_to_any.detect_modality("Hello world") == 'text'
log.success("✓ Detects text")

assert flux_to_any.detect_modality([[1, 2], [3, 4]]) == 'grid'
log.success("✓ Detects grid (list)")

assert flux_to_any.detect_modality(torch.tensor([[1, 2], [3, 4]], dtype=torch.long)) == 'grid'
log.success("✓ Detects grid (tensor)")

assert flux_to_any.detect_modality(torch.randn(1000)) == 'audio'
log.success("✓ Detects audio")

# Test grid encoding
grid = [[1, 2, 3], [4, 5, 6]]
wave = flux_to_any.encode(grid, modality='grid')
assert wave.shape == torch.Size([432])
log.success(f"✓ Grid encoding: {wave.shape}")

# Test grid decoding
decoded = flux_to_any.decode(wave, modality='grid', grid_size=(2, 3))
assert decoded.shape == torch.Size([2, 3])
log.success(f"✓ Grid decoding: {decoded.shape}")

# Test image decoding
image = flux_to_any.decode(wave, modality='image', size=(64, 64))
assert image.shape == torch.Size([64, 64, 3])
log.success(f"✓ Image decoding: {image.shape}")

# Test cross-modal: grid → image
result = flux_to_any([[1, 0, 1], [0, 1, 0], [1, 0, 1]], output_modality='image', size=(64, 64))
assert result.output.shape == torch.Size([64, 64, 3])
log.success(f"✓ Cross-modal grid → image")

log.cell_end("Cell 10 — Test 3: FluxToAny", "PASS")

## Cell 11: Demo 1 — Cross-Modal Processing

In [ ]:
log.cell_start("Cell 11 — Demo 1: Cross-Modal")

import matplotlib.pyplot as plt

# Demo: Grid → Image
log.info("Grid → Image conversion:")

grids = [
    ("Checkerboard", [[1, 0, 1], [0, 1, 0], [1, 0, 1]]),
    ("Diagonal", [[1, 0, 0], [0, 2, 0], [0, 0, 3]]),
    ("Cross", [[0, 1, 0], [1, 1, 1], [0, 1, 0]]),
    ("Border", [[1, 1, 1], [1, 0, 1], [1, 1, 1]]),
]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i, (name, grid) in enumerate(grids):
    # Show grid
    grid_tensor = torch.tensor(grid, dtype=torch.float32)
    axes[0, i].imshow(grid_tensor, cmap='tab10', vmin=0, vmax=9)
    axes[0, i].set_title(f"Grid: {name}")
    axes[0, i].axis('off')
    
    # Convert to image
    result = flux_to_any(grid, output_modality='image', size=(128, 128), style='dream')
    axes[1, i].imshow(result.output.cpu().numpy())
    axes[1, i].set_title(f"Image: {name}")
    axes[1, i].axis('off')

plt.suptitle("Grid → Wave → Image (Cross-Modal Processing)", fontsize=14)
plt.tight_layout()
plt.savefig('/kaggle/working/demo_cross_modal.png', dpi=150)
plt.show()

log.success("Demo saved to demo_cross_modal.png")
log.cell_end("Cell 11 — Demo 1: Cross-Modal", "PASS")

## Cell 12: Demo 2 — 3 Physics Engines Comparison

In [ ]:
log.cell_start("Cell 12 — Demo 2: 3 Physics Engines")

# Create waves for different concepts
torch.manual_seed(123)
waves = {
    'Cosmic': torch.randn(432, device=device) * 2,
    'Ocean': torch.randn(432, device=device) * 2,
    'Fire': torch.randn(432, device=device) * 2,
}

fig, axes = plt.subplots(3, 4, figsize=(16, 12))

size = (128, 128)

for row, (concept, wave) in enumerate(waves.items()):
    # Pure Gravity
    img_g = wave_to_image.gravity(wave, size[0], size[1])
    axes[row, 0].imshow(img_g.cpu().numpy())
    axes[row, 0].set_title(f"{concept}\nGravity" if row == 0 else "Gravity")
    axes[row, 0].axis('off')
    
    # Pure Interference
    img_i = wave_to_image.interference(wave, size[0], size[1])
    axes[row, 1].imshow(img_i.cpu().numpy())
    axes[row, 1].set_title(f"{concept}\nInterference" if row == 0 else "Interference")
    axes[row, 1].axis('off')
    
    # Pure Thermodynamic
    img_t = wave_to_image.thermodynamic(wave, size[0], size[1])
    axes[row, 2].imshow(img_t.cpu().numpy())
    axes[row, 2].set_title(f"{concept}\nThermodynamic" if row == 0 else "Thermodynamic")
    axes[row, 2].axis('off')
    
    # Blended (Dream style)
    img_blend = wave_to_image.decode(wave, size=size, style='dream')
    axes[row, 3].imshow(img_blend.cpu().numpy())
    axes[row, 3].set_title(f"{concept}\nDream Blend" if row == 0 else "Dream Blend")
    axes[row, 3].axis('off')

plt.suptitle("3 Physics Engines: Gravity | Interference | Thermodynamic | Blend", fontsize=14)
plt.tight_layout()
plt.savefig('/kaggle/working/demo_3_physics.png', dpi=150)
plt.show()

log.success("Demo saved to demo_3_physics.png")
log.cell_end("Cell 12 — Demo 2: 3 Physics Engines", "PASS")

## Cell 13: Demo 3 — All Style Presets

In [ ]:
log.cell_start("Cell 13 — Demo 3: Style Presets")

# Generate with all style presets
torch.manual_seed(456)
wave = torch.randn(432, device=device) * 2

fig, axes = plt.subplots(1, 5, figsize=(20, 4))

for i, style_name in enumerate(STYLE_PRESETS.keys()):
    preset = STYLE_PRESETS[style_name]
    img = wave_to_image.decode(wave, size=(128, 128), style=style_name)
    
    axes[i].imshow(img.cpu().numpy())
    axes[i].set_title(f"{style_name.capitalize()}\n({preset.gravity:.1f}, {preset.interference:.1f}, {preset.thermodynamic:.1f})")
    axes[i].axis('off')

plt.suptitle("5 Style Presets: (Gravity, Interference, Thermodynamic)", fontsize=14)
plt.tight_layout()
plt.savefig('/kaggle/working/demo_styles.png', dpi=150)
plt.show()

log.success("Demo saved to demo_styles.png")
log.cell_end("Cell 13 — Demo 3: Style Presets", "PASS")

## Cell 14: Save Flux-X-complete.flx

Create the complete Phase 8.9 model with all adapters and upload to HuggingFace.

In [ ]:
log.cell_start("Cell 14 — Save Flux-X-complete.flx")

from datetime import datetime

print('\n  Building Flux-X-complete.flx (Phase 8.9 with all adapters)...\n')

# ─────────────────────────────────────────────
# Build the new .flx archive with adapters
# ─────────────────────────────────────────────

flx_complete = {
    'format': 'FLUX',
    'version': '1.2-complete',  # Complete = all modality adapters
    
    # Metadata
    'metadata': {
        'source_phase': '8.9',
        'created': datetime.now().isoformat(),
        'description': 'FLUX with complete modality suite (text, grid, image, audio)',
        'base_model': 'Flux-X.flx',
        'adapters': ['text', 'grid', 'image', 'audio'],
    },
}

# Copy all sections from the loaded model
print('  Extracting model components...')

# CSE
flx_complete['cse'] = {
    'config': {'wave_dim': 432},
    'state_dict': model.cse.state_dict(),
}
print('    ✓ CSE')

# Field
flx_complete['field'] = {
    'config': {'h': 96, 'w': 96, 'd': 96, 'features': 512},
    'state_dict': model.field.state_dict(),
}
if hasattr(model, 'gr'):
    flx_complete['field']['gravity_state'] = model.gr.save_state()
if hasattr(model, 'tl'):
    flx_complete['field']['thermodynamic_state'] = model.tl.save_state()
print('    ✓ Field + GR + TL')

# Memory
flx_complete['memory'] = {}
if hasattr(model, 'working_memory'):
    flx_complete['memory']['working'] = model.working_memory.state_dict_memory()
if hasattr(model, 'episodic_memory'):
    flx_complete['memory']['episodic'] = model.episodic_memory.save_state()
if hasattr(model, 'semantic_memory'):
    flx_complete['memory']['semantic'] = model.semantic_memory.save_state()
episodic_count = model.episodic_memory.size if hasattr(model, 'episodic_memory') else 0
print(f'    ✓ Memory ({episodic_count} episodic entries)')

# Decoder (clean _orig_mod. prefix)
decoder_state = model.decoder.state_dict()
cleaned_decoder = {k.replace('_orig_mod.', ''): v for k, v in decoder_state.items()}
flx_complete['decoder'] = {
    'config': {'hidden_dim': 1024, 'num_layers': 4},
    'state_dict': cleaned_decoder,
}
print('    ✓ WaveDecoder')

# Causal
flx_complete['causal'] = {}
if hasattr(model, 'cgn'):
    flx_complete['causal']['cgn_state'] = model.cgn.save_state()
if hasattr(model, 'causal_graph'):
    flx_complete['causal']['graph_state'] = model.causal_graph.save_state()
print('    ✓ CGN + CausalGraph')

# Bridges
flx_complete['bridges'] = {}
if hasattr(model, 'wave_to_field'):
    flx_complete['bridges']['wave_to_field'] = model.wave_to_field.state_dict()
if hasattr(model, 'field_to_wave'):
    flx_complete['bridges']['field_to_wave'] = model.field_to_wave.state_dict()
if hasattr(model, 'memory_router'):
    flx_complete['bridges']['router'] = model.memory_router.save_state()
if hasattr(model, 'output_head'):
    flx_complete['bridges']['output_head'] = model.output_head.state_dict()
print('    ✓ Bridges')

# ─────────────────────────────────────────────
# Add all adapters (Phase 8.8 + Phase 8.9)
# ─────────────────────────────────────────────

flx_complete['adapters'] = {
    'version': '8.9',
    
    # Grid adapters (from Phase 8.8)
    'grid_to_wave': flux_to_any.input_adapters['grid'].state_dict(),
    'wave_to_grid': flux_to_any.output_adapters['grid'].state_dict(),
    
    # Image adapters (Phase 8.9 - 3 physics engines)
    'wave_to_image': flux_to_any.output_adapters['image'].state_dict(),
    
    # Audio adapters (Phase 8.9 - stubs)
    'audio_to_wave': flux_to_any.input_adapters['audio'].state_dict(),
    'wave_to_audio': flux_to_any.output_adapters['audio'].state_dict(),
}
print('    ✓ WaveToX Adapters (grid, image, audio)')

# ─────────────────────────────────────────────
# Save locally
# ─────────────────────────────────────────────

output_path = CHECKPOINT_DIR / 'Flux-X-complete.flx'
torch.save(flx_complete, str(output_path))
size_mb = output_path.stat().st_size / 1e6
print(f'\n  ✓ Saved: {output_path.name} ({size_mb:.1f} MB)')

log.cell_end("Cell 14 — Save Flux-X-complete.flx", "PASS")

## Cell 15: Upload to HuggingFace Hub

In [ ]:
log.cell_start("Cell 15 — Upload to HuggingFace")

print('\n  Uploading to HuggingFace Hub...')

from huggingface_hub import HfApi
from datetime import datetime

if hf_token:
    try:
        api = HfApi(token=hf_token)
        
        # Upload the file
        api.upload_file(
            path_or_fileobj=str(output_path),
            path_in_repo='checkpoints/Flux-X-complete.flx',
            repo_id='UnseenGAP/FLUX',
            commit_message=f'Add Flux-X-complete.flx (Phase 8.9 complete modality suite) — {datetime.now().isoformat()}',
        )
        
        print(f'  ✓ Uploaded to HuggingFace Hub')
        print(f'    https://huggingface.co/UnseenGAP/FLUX/blob/main/checkpoints/Flux-X-complete.flx')
        log.success("Flux-X-complete.flx uploaded to HuggingFace")
        
        # Upload demo images if they exist
        demo_dir = ROOT / 'output'
        demo_dir.mkdir(exist_ok=True)
        
        for demo_file in ['demo_cross_modal.png', 'demo_3_physics.png', 'demo_styles.png']:
            # Check both possible locations
            demo_path = ROOT / demo_file
            if not demo_path.exists():
                demo_path = Path('/kaggle/working') / demo_file
            
            if demo_path.exists():
                api.upload_file(
                    path_or_fileobj=str(demo_path),
                    path_in_repo=f'demos/phase8_9/{demo_file}',
                    repo_id='UnseenGAP/FLUX',
                )
                log.info(f"Uploaded {demo_file}")
        
        log.success("All uploads complete")
        
    except Exception as e:
        print(f'  ⚠ Upload failed: {e}')
        print(f'    File saved locally: {output_path}')
        log.warning(f"HF upload failed: {str(e)[:50]}")
else:
    log.warning("No HF_TOKEN - skipping upload")
    log.info(f"Model saved locally at: {output_path}")

log.cell_end("Cell 15 — Upload to HuggingFace", "PASS")

## Cell 16: Summary

In [ ]:
log.separator("Phase 8.9 Complete: Universal Modality Suite")

print("""
╔════════════════════════════════════════════════════════════════════╗
║                     PHASE 8.9 COMPLETE                            ║
╠════════════════════════════════════════════════════════════════════╣
║                                                                    ║
║  NEW ADAPTERS:                                                     ║
║  ├── ImageToWave: Patch-based image encoder                       ║
║  ├── WaveToImage_Universal: 3 physics-based renderers             ║
║  │   ├── Gravity: Mass attractors → smooth gradients              ║
║  │   ├── Interference: Wave superposition → ripples               ║
║  │   └── Thermodynamic: Energy minimization → textures            ║
║  ├── AudioToWave: Audio encoder (stub)                            ║
║  └── WaveToAudio: Audio decoder (stub)                            ║
║                                                                    ║
║  UNIFIED MODEL:                                                    ║
║  └── FluxToAny: Universal interface for all modalities            ║
║      • Text → Wave → Text/Grid/Image                              ║
║      • Grid → Wave → Text/Grid/Image                              ║
║      • Audio → Wave (stub)                                         ║
║                                                                    ║
║  STYLE PRESETS:                                                    ║
║  ├── photorealistic: (0.7, 0.2, 0.1)                              ║
║  ├── abstract: (0.2, 0.6, 0.2)                                    ║
║  ├── crystalline: (0.1, 0.3, 0.6)                                 ║
║  ├── organic: (0.4, 0.1, 0.5)                                     ║
║  └── dream: (0.33, 0.33, 0.34)                                    ║
║                                                                    ║
║  OUTPUT: Flux-X-complete.flx                                       ║
║  • Format version: 1.2-complete                                    ║
║  • All adapters included                                           ║
║                                                                    ║
╚════════════════════════════════════════════════════════════════════╝
""")

log.success("Phase 8.9 notebook complete!")
print("\nNext steps:")
print("  1. Run train_grid_adapters.py on ARC dataset")
print("  2. Implement full audio adapters (Phase 10)")
print("  3. Scale up image generation resolution")